# ARV Model: Quartile Rehab Cost

In [4]:
# ============================================================
# CLEAN START — Real Estate Portfolio Optimization (Gurobi)
# ============================================================

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# ----------------------------
# 1) CONFIG (EXPLICIT HEADERS)
# ----------------------------

CSV_PATH = "Quartile Update Costs.csv"

BUDGET = 5_000_000
ROI_TARGET = 0.15
SFH_MIN_SHARE = 0.10
SFH_MAX_SHARE = 0.30

# ---- Exact column names from your CSV
COL_ID = "MLS"
COL_ARV = "Resale Price"
COL_PURCHASE = "Purchase Price"
COL_REHAB = "Rehab Costs"
COL_OTHER = "Closing Costs"
COL_SFH_FLAG = "Condo/SFH"   # 1 = SFH, 0 = Condo


# ----------------------------
# 2) LOAD DATA
# ----------------------------

df = pd.read_csv(CSV_PATH)

# Ensure numeric columns
for c in [COL_ARV, COL_PURCHASE, COL_REHAB, COL_OTHER]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[COL_ARV, COL_PURCHASE, COL_REHAB, COL_OTHER]).reset_index(drop=True)

# Convert SFH flag to numeric (1 = SFH, 0 = Condo)
df[COL_SFH_FLAG] = pd.to_numeric(df[COL_SFH_FLAG], errors="coerce").fillna(0).astype(int)

# ----------------------------
# 3) CALCULATE COST & PROFIT
# ----------------------------

df["Total_Investment"] = df[COL_PURCHASE] + df[COL_REHAB] + df[COL_OTHER]
df["Profit"] = df[COL_ARV] - df["Total_Investment"]

# Drop invalid rows
df = df[df["Total_Investment"] > 0].reset_index(drop=True)

# Enforce ROI filter: ARV >= 1.15 * Total Investment
df = df[df[COL_ARV] >= (1 + ROI_TARGET) * df["Total_Investment"]].reset_index(drop=True)

if len(df) == 0:
    raise ValueError("No properties meet the 15% ROI requirement.")

# Index sets
I = df.index.tolist()
S = df.index[df[COL_SFH_FLAG] == 1].tolist()

if len(S) == 0:
    raise ValueError("No Single Family properties found (Condo/SFH must equal 1 for SFH).")

# ----------------------------
# 4) BUILD GUROBI MODEL
# ----------------------------

m = gp.Model("RealEstatePortfolio")

# Decision variable: Buy property i
buy = m.addVars(I, vtype=GRB.BINARY, name="Buy")

# Objective: Maximize total profit
m.setObjective(
    gp.quicksum(df.loc[i, "Profit"] * buy[i] for i in I),
    GRB.MAXIMIZE
)

# Budget constraint
total_invest = gp.quicksum(df.loc[i, "Total_Investment"] * buy[i] for i in I)
m.addConstr(total_invest <= BUDGET, name="Budget")

# SFH share constraints
sfh_invest = gp.quicksum(df.loc[i, "Total_Investment"] * buy[i] for i in S)

m.addConstr(sfh_invest >= SFH_MIN_SHARE * total_invest, name="SFH_min")
m.addConstr(sfh_invest <= SFH_MAX_SHARE * total_invest, name="SFH_max")

# Prevent trivial "buy nothing"
m.addConstr(total_invest >= 1, name="BuyAtLeastOne")

# Solve
m.optimize()

# ----------------------------
# 5) RESULTS
# ----------------------------

if m.Status == GRB.OPTIMAL:

    chosen = [i for i in I if buy[i].X > 0.5]
    selected = df.loc[chosen].copy()

    total_inv = sum(df.loc[i, "Total_Investment"] * buy[i].X for i in I)
    total_profit = sum(df.loc[i, "Profit"] * buy[i].X for i in I)
    sfh_inv = sum(df.loc[i, "Total_Investment"] * buy[i].X for i in S)

    print("\n========== OPTIMAL PORTFOLIO ==========")
    print(f"Properties selected: {len(selected)}")
    print(f"Total invested: ${total_inv:,.0f}")
    print(f"Total profit: ${total_profit:,.0f}")
    print(f"Portfolio ROI: {total_profit/total_inv:.2%}")
    print(f"SFH Share: {sfh_inv/total_inv:.2%}")
    print("=======================================\n")

    print(selected[[COL_ID, COL_ARV, COL_PURCHASE, COL_REHAB, COL_OTHER, "Total_Investment", "Profit"]])
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 285H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 4 rows, 29 columns and 116 nonzeros (Max)
Model fingerprint: 0x216b94d8
Model has 29 linear objective coefficients
Variable types: 0 continuous, 29 integer (29 binary)
Coefficient statistics:
  Matrix range     [2e+04, 1e+06]
  Objective range  [4e+04, 3e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+06]

Presolve removed 0 rows and 1 columns
Presolve time: 0.00s
Presolved: 4 rows, 28 columns, 112 nonzeros
Variable types: 0 continuous, 28 integer (27 binary)
Found heuristic solution: objective 898842.63688
Found heuristic solution: objective 1027370.1569
Found heuristic solution: objective 1096962.3360
Found heuristic solution: objective 1104348.9809

Root relaxation: objective 1.623145e+06, 3 iterations, 0.

In [7]:
# ============================================================
# Maximize PORTFOLIO ROI (Profit / Invest) with Gurobi
# - NO per-deal ARV/ROI constraints (you asked to remove them)
# - Budget (cash): total investment <= $5,000,000
# - SFH dollars between 10% and 30% of total invested dollars
#
# IMPORTANT: Maximizing ROI is a *fractional* objective.
# We solve it with Dinkelbach's algorithm:
#   repeatedly solve: maximize (TotalProfit - r * TotalInvest)
#   then update r = TotalProfit / TotalInvest until convergence.
# ============================================================

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# ----------------------------
# 1) CONFIG (EXPLICIT HEADERS)
# ----------------------------
CSV_PATH = "Quartile Update Costs.csv"

BUDGET = 5_000_000
SFH_MIN_SHARE = 0.10
SFH_MAX_SHARE = 0.30

# Exact column names from your CSV
COL_ID       = "MLS"
COL_ARV      = "Resale Price"
COL_PURCHASE = "Purchase Price"
COL_REHAB    = "Rehab Costs"
COL_OTHER    = "Closing Costs"
COL_SFH_FLAG = "Condo/SFH"   # 1 = SFH, 0 = Condo

# Dinkelbach settings
MAX_ITERS = 50
TOL = 1e-6  # convergence tolerance on (Profit - r*Invest)

# ----------------------------
# 2) LOAD + CLEAN DATA
# ----------------------------
df = pd.read_csv(CSV_PATH)

# numeric coercion
for c in [COL_ARV, COL_PURCHASE, COL_REHAB, COL_OTHER]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[COL_SFH_FLAG] = pd.to_numeric(df[COL_SFH_FLAG], errors="coerce").fillna(0).astype(int)

df = df.dropna(subset=[COL_ARV, COL_PURCHASE, COL_REHAB, COL_OTHER]).reset_index(drop=True)

# investment + profit
df["Total_Investment"] = df[COL_PURCHASE] + df[COL_REHAB] + df[COL_OTHER]
df["Profit"] = df[COL_ARV] - df["Total_Investment"]

# basic sanity filters
df = df[df["Total_Investment"] > 0].reset_index(drop=True)

I = df.index.tolist()
S = df.index[df[COL_SFH_FLAG] == 1].tolist()

if len(I) == 0:
    raise ValueError("No valid rows after cleaning.")
if len(S) == 0:
    raise ValueError("No SFH rows found. Ensure Condo/SFH uses 1 for SFH and 0 for Condo.")

# ----------------------------
# 3) BUILD BASE MODEL (constraints fixed; objective changes each iteration)
# ----------------------------
m = gp.Model("MaxPortfolioROI")
m.Params.OutputFlag = 1

buy = m.addVars(I, vtype=GRB.BINARY, name="Buy")

total_invest = gp.quicksum(df.loc[i, "Total_Investment"] * buy[i] for i in I)
total_profit = gp.quicksum(df.loc[i, "Profit"] * buy[i] for i in I)

# Budget
m.addConstr(total_invest <= BUDGET, name="Budget")

# SFH share (by invested dollars)
sfh_invest = gp.quicksum(df.loc[i, "Total_Investment"] * buy[i] for i in S)
m.addConstr(sfh_invest >= SFH_MIN_SHARE * total_invest, name="SFH_min")
m.addConstr(sfh_invest <= SFH_MAX_SHARE * total_invest, name="SFH_max")

# Prevent "buy nothing"
m.addConstr(total_invest >= 1.0, name="BuyAtLeastOne")
#m.addConstr(total_invest >= 4_500_000, name="MinSpend")

# ----------------------------
# 4) DINKELBACH ITERATIONS
# ----------------------------
r = 0.0  # starting ROI guess
best_solution = None

for it in range(1, MAX_ITERS + 1):
    # Maximize Profit - r * Invest
    m.setObjective(total_profit - r * total_invest, GRB.MAXIMIZE)
    m.optimize()

    if m.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Gurobi did not find an optimal solution (status={m.Status}).")

    invest_val = total_invest.getValue()
    profit_val = total_profit.getValue()

    # portfolio ROI
    roi_val = profit_val / invest_val if invest_val > 0 else float("-inf")

    # Dinkelbach gap
    gap = profit_val - r * invest_val

    print(f"\n--- Iter {it} ---")
    print(f"Invest: ${invest_val:,.2f}")
    print(f"Profit: ${profit_val:,.2f}")
    print(f"ROI:    {roi_val:.6f}")
    print(f"Gap:    {gap:.8f}")

    # Save best solution so far
    chosen = [i for i in I if buy[i].X > 0.5]
    best_solution = (roi_val, invest_val, profit_val, chosen)

    # Convergence check
    if abs(gap) <= TOL:
        print("\nConverged ✅")
        break

    # Update r
    r = roi_val

# ----------------------------
# 5) REPORT RESULTS
# ----------------------------
roi_val, invest_val, profit_val, chosen = best_solution

selected = df.loc[chosen].copy()
selected["Selected"] = 1
selected["Item_ROI"] = selected["Profit"] / selected["Total_Investment"]

sfh_invest_val = selected.loc[selected[COL_SFH_FLAG] == 1, "Total_Investment"].sum()
sfh_share = sfh_invest_val / invest_val if invest_val > 0 else 0

print("\n========== BEST ROI PORTFOLIO ==========")
print(f"Selected properties: {len(selected)}")
print(f"Total invested:      ${invest_val:,.0f}")
print(f"Total profit:        ${profit_val:,.0f}")
print(f"Portfolio ROI:       {roi_val:.2%}")
print(f"SFH invested:        ${sfh_invest_val:,.0f} ({sfh_share:.2%})")
print("=======================================\n")

# Output a clean table + save
cols_out = [COL_ID, COL_SFH_FLAG, COL_ARV, COL_PURCHASE, COL_REHAB, COL_OTHER,
            "Total_Investment", "Profit", "Item_ROI"]
cols_out = [c for c in cols_out if c in selected.columns]
selected = selected[cols_out].sort_values(by="Item_ROI", ascending=False)

print(selected.to_string(index=False))

out_path = "gurobi_max_roi_selected.xlsx"
selected.to_excel(out_path, index=False)
print(f"\nSaved: {out_path}")

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 285H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 4 rows, 103 columns and 412 nonzeros (Max)
Model fingerprint: 0x6cf1c614
Model has 103 linear objective coefficients
Variable types: 0 continuous, 103 integer (103 binary)
Coefficient statistics:
  Matrix range     [2e+04, 2e+06]
  Objective range  [2e+03, 3e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+06]

Presolve removed 0 rows and 4 columns
Presolve time: 0.00s
Presolved: 4 rows, 99 columns, 396 nonzeros
Variable types: 0 continuous, 99 integer (96 binary)
Found heuristic solution: objective 614265.48000
Found heuristic solution: objective 1029438.8369

Root relaxation: objective 1.625078e+06, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current No